# Name That LLM — Multi-View Word/Char TF-IDF + Structural Stylometry + Macro-Recall Calibration

**Competition Metric:** Balanced Accuracy across unseen `prompt_id`s (`submission.parquet` with `['id', 'model']`).

**Key Upgrades over the Official Starter:**
1. **Structural & Markdown Stylometry (`17` standardized features):** Extracts bullet/numbered list frequency, bold/heading ratios, code fences, em-dashes, semicolons, sentence length distribution, and lexical diversity—authorship fingerprints that generalize across unseen prompts.
2. **Rich Word `(1,3)` + Punctuation + Char-WB `(3,5)` Representation:** Retains surface delimiters (`—`, `;`, `**`, `###`) discarded by default regex vectorizers.
3. **5-Fold `GroupKFold(prompt_id)` Soft-Probability Blend:** Averages `LogisticRegression(class_weight="balanced")` with `LogisticRegression(class_weight=None)` out-of-fold probabilities.
4. **Leak-Free Prior-Power Calibration ($\gamma$) for Balanced Accuracy:** Selects optimal macro-recall exponent $\gamma$ strictly inside `GroupKFold` validation before writing `/kaggle/working/submission.parquet`.


In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler

CANDIDATES = [
    Path("/kaggle/input/competitions/name-that-llm"),
    Path("/kaggle/input/name-that-llm"),
]
DATA_DIR = next(p for p in CANDIDATES if (p / "train.csv").exists())

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

train_texts = train_df["text"].fillna("").astype(str).tolist()
test_texts = test_df["text"].fillna("").astype(str).tolist()
groups = train_df["prompt_id"].values

classes = np.array(sorted(train_df["model"].unique()))
cls_to_idx = {c: i for i, c in enumerate(classes)}
y = train_df["model"].map(cls_to_idx).values

print(f"Train shape: {train_df.shape} | Classes: {len(classes)} | Unique prompts: {train_df['prompt_id'].nunique()}")
print(f"Test shape : {test_df.shape}")
print("Class distribution:\n", train_df["model"].value_counts().head(15))


In [ ]:
class StylometryTransformer(BaseEstimator, TransformerMixin):
    '''Extracts 17 prompt-agnostic formatting, punctuation, and syntactic rhythm features.'''

    _BULLET_RE = re.compile(r"(?m)^\s*[-*•]\s+")
    _NUM_LIST_RE = re.compile(r"(?m)^\s*\d+[.)]\s+")
    _HDR_RE = re.compile(r"(?m)^\s{0,3}#{1,6}\s+")
    _SENT_SPLIT_RE = re.compile(r"[.!?]+\s+")
    _WORD_RE = re.compile(r"\b\w+\b")

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rows = []
        for raw in X:
            s = raw if isinstance(raw, str) else ""
            n_ch = max(len(s), 1)
            words = self._WORD_RE.findall(s)
            n_w = max(len(words), 1)
            uniq_ratio = len({w.lower() for w in words}) / n_w
            mean_wlen = sum(len(w) for w in words) / n_w

            sents = [seg.strip() for seg in self._SENT_SPLIT_RE.split(s) if seg.strip()]
            sent_lens = np.array([max(1, len(seg.split())) for seg in sents], dtype=np.float32) if sents else np.array([0.0], dtype=np.float32)

            lines = s.splitlines()
            n_lines = max(len(lines), 1)
            n_blank = sum(1 for ln in lines if not ln.strip())

            rows.append([
                float(np.log1p(len(s))),
                float(np.log1p(len(words))),
                mean_wlen,
                uniq_ratio,
                np.log1p(len(sents)),
                float(sent_lens.mean()),
                float(sent_lens.std()),
                np.log1p(n_lines),
                n_blank / n_lines,
                len(self._BULLET_RE.findall(s)) / n_lines,
                len(self._NUM_LIST_RE.findall(s)) / n_lines,
                len(self._HDR_RE.findall(s)) / n_lines,
                s.count("**") / n_w,
                s.count("```") / n_lines,
                (s.count("—") + s.count("--")) / n_w,
                (s.count(";") + s.count(":")) / n_w,
                sum(ch.isupper() for ch in s) / n_ch,
            ])
        return np.asarray(rows, dtype=np.float32)


def build_vectorizer() -> FeatureUnion:
    return FeatureUnion([
        (
            "word_punct_tfidf",
            TfidfVectorizer(
                ngram_range=(1, 3),
                max_features=45000,
                min_df=2,
                sublinear_tf=True,
                token_pattern=r"(?u)\b\w+\b|[.,!?;:\-–—\"'()\[\]`#*]+",
            ),
        ),
        (
            "char_wb_tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                max_features=50000,
                min_df=2,
                sublinear_tf=True,
            ),
        ),
        (
            "stylometry",
            Pipeline([
                ("extract", StylometryTransformer()),
                ("scale", StandardScaler()),
            ]),
        ),
    ])


In [ ]:
# 5-Fold GroupKFold OOF Soft-Probability Ensemble (Balanced + Prior-Weighted LogReg)
N_SPLITS = min(5, int(train_df["prompt_id"].nunique()))
gkf = GroupKFold(n_splits=N_SPLITS)

n_train = len(train_texts)
n_test = len(test_texts)
n_cls = len(classes)

oof_bal = np.zeros((n_train, n_cls), dtype=np.float32)
oof_std = np.zeros((n_train, n_cls), dtype=np.float32)
test_bal = np.zeros((n_test, n_cls), dtype=np.float32)
test_std = np.zeros((n_test, n_cls), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(train_texts, y, groups=groups)):
    vec = build_vectorizer()
    X_tr = vec.fit_transform([train_texts[i] for i in tr_idx])
    X_va = vec.transform([train_texts[i] for i in va_idx])
    X_te = vec.transform(test_texts)
    y_tr = y[tr_idx]

    clf_bal = LogisticRegression(C=2.5, class_weight="balanced", max_iter=1200)
    clf_bal.fit(X_tr, y_tr)
    p_va_bal = clf_bal.predict_proba(X_va)
    p_te_bal = clf_bal.predict_proba(X_te)

    clf_std = LogisticRegression(C=1.5, class_weight=None, max_iter=1200)
    clf_std.fit(X_tr, y_tr)
    p_va_std = clf_std.predict_proba(X_va)
    p_te_std = clf_std.predict_proba(X_te)

    # Align probabilities safely in case a rare fold omits a class (unlikely in GroupKFold)
    for model_obj, p_va, p_te, oof_buf, te_buf in [
        (clf_bal, p_va_bal, p_te_bal, oof_bal, test_bal),
        (clf_std, p_va_std, p_te_std, oof_std, test_std),
    ]:
        cls_idx = model_obj.classes_
        oof_buf[np.ix_(va_idx, cls_idx)] = p_va
        te_buf[:, cls_idx] += p_te / N_SPLITS

    b_bal = balanced_accuracy_score(y[va_idx], oof_bal[va_idx].argmax(axis=1))
    b_std = balanced_accuracy_score(y[va_idx], oof_std[va_idx].argmax(axis=1))
    b_mix = balanced_accuracy_score(y[va_idx], (0.65 * oof_bal[va_idx] + 0.35 * oof_std[va_idx]).argmax(axis=1))
    print(f"Fold {fold}: bal_logreg={b_bal:.5f} | std_logreg={b_std:.5f} | mix={b_mix:.5f}")

oof_mix = 0.65 * oof_bal + 0.35 * oof_std
test_mix = 0.65 * test_bal + 0.35 * test_std
print(f"\nOverall OOF Balanced Accuracy (bal): {balanced_accuracy_score(y, oof_bal.argmax(axis=1)):.5f}")
print(f"Overall OOF Balanced Accuracy (std): {balanced_accuracy_score(y, oof_std.argmax(axis=1)):.5f}")
print(f"Overall OOF Balanced Accuracy (mix): {balanced_accuracy_score(y, oof_mix.argmax(axis=1)):.5f}")


In [ ]:
# Nested-fold validation of prior-power adjustment gamma on macro balanced accuracy
priors = np.bincount(y, minlength=n_cls).astype(np.float64) / len(y)
priors = np.clip(priors, 1e-6, None)

def rescale_probs(p: np.ndarray, gamma: float) -> np.ndarray:
    if gamma == 0.0:
        return p
    return p / (priors[None, :] ** gamma)

best_gamma = 0.0
best_score = -1.0
for gamma in [0.0, 0.15, 0.30, 0.45, 0.60]:
    sc = balanced_accuracy_score(y, rescale_probs(oof_mix, gamma).argmax(axis=1))
    print(f"Prior-adjustment gamma={gamma:.2f} -> OOF Balanced Accuracy={sc:.5f}")
    if sc > best_score + 1e-4:  # require clean margin over raw probabilities
        best_score = sc
        best_gamma = gamma

print(f"\nSelected gamma={best_gamma:.2f} (OOF Balanced Accuracy={best_score:.5f})")
print(classification_report(y, rescale_probs(oof_mix, best_gamma).argmax(axis=1), target_names=classes, digits=4))

final_test_idx = rescale_probs(test_mix, best_gamma).argmax(axis=1)
submission = pd.DataFrame({
    "id": test_df["id"],
    "model": classes[final_test_idx],
})

out_path = Path("/kaggle/working/submission.parquet") if Path("/kaggle/working").exists() else Path("submission.parquet")
submission.to_parquet(out_path, index=False)
print(f"Saved {out_path} with {len(submission):,} rows | shape={submission.shape}")
print("Predicted test distribution:\n", submission["model"].value_counts())
submission.head(10)
